# Legal Document Summarization Evaluation

This notebook provides comprehensive evaluation of legal document summarization methods using domain-specific metrics.

In [ ]:
# Import necessary libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import json
from collections import defaultdict

# Import our custom modules
from data_loader import LegalDataLoader
from legal_processor import LegalTextProcessor
from extractive_summarizer import LegalExtractiveSummarizer
from abstractive_summarizer import LegalAbstractiveSummarizer
from hybrid_summarizer import LegalHybridSummarizer
from evaluation import LegalSummarizationEvaluator, quick_evaluate

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

ImportError: attempted relative import with no known parent package

## 1. Load Test Dataset

In [ ]:
# Load test data
data_loader = LegalDataLoader()
test_data = data_loader.load_in_abs_dataset(split='test')

print(f"Loaded {len(test_data['judgments'])} test samples")

# Use a subset for detailed evaluation
test_size = 50
test_documents = test_data['judgments'][:test_size]
test_summaries = test_data['summaries'][:test_size]

print(f"Using {test_size} samples for evaluation")
print(f"Average document length: {np.mean([len(doc.split()) for doc in test_documents]):.1f} words")
print(f"Average summary length: {np.mean([len(summary.split()) for summary in test_summaries]):.1f} words")

## 2. Initialize Summarization Methods

In [ ]:
# Initialize all summarization methods
legal_processor = LegalTextProcessor()
extractive_summarizer = LegalExtractiveSummarizer(legal_processor)
abstractive_summarizer = LegalAbstractiveSummarizer(model_name='t5-base')
hybrid_summarizer = LegalHybridSummarizer(
    extractive_summarizer=extractive_summarizer,
    abstractive_summarizer=abstractive_summarizer,
    legal_processor=legal_processor
)
evaluator = LegalSummarizationEvaluator(legal_processor)

print("All summarization methods initialized!")

## 3. Generate Summaries Using All Methods

In [ ]:
# Generate summaries using all methods
print("Generating summaries using all methods...")

results = {
    'extractive': [],
    'abstractive': [],
    'hybrid': []
}

for i, (document, reference) in enumerate(zip(test_documents, test_summaries)):
    if i % 10 == 0:
        print(f"Processing document {i+1}/{test_size}")
    
    # Extractive summarization
    ext_result = extractive_summarizer.generate_extractive_summary(
        document,
        num_sentences=5,
        method='textrank'
    )
    results['extractive'].append(ext_result['summary'])
    
    # Abstractive summarization
    abs_result = abstractive_summarizer.generate_abstractive_summary(
        document,
        max_length=200,
        min_length=50
    )
    results['abstractive'].append(abs_result['summary'])
    
    # Hybrid summarization
    hyb_result = hybrid_summarizer.optimize_hybrid_output(
        document,
        target_length=200
    )
    
    if 'error' not in hyb_result:
        results['hybrid'].append(hyb_result['summary'])
    else:
        results['hybrid'].append('')  # Fallback for errors

print("Summary generation completed!")
print(f"Generated {len(results['extractive'])} extractive summaries")
print(f"Generated {len(results['abstractive'])} abstractive summaries")
print(f"Generated {len(results['hybrid'])} hybrid summaries")

## 4. Evaluate Each Method

In [ ]:
# Evaluate each method
evaluation_results = {}

for method_name, generated_summaries in results.items():
    print(f"\nEvaluating {method_name} method...")
    
    # Filter out empty summaries
valid_indices = [i for i, summary in enumerate(generated_summaries) if summary.strip()]
    valid_generated = [generated_summaries[i] for i in valid_indices]
    valid_reference = [test_summaries[i] for i in valid_indices]
    valid_documents = [test_documents[i] for i in valid_indices]
    
    if valid_generated:
        eval_result = evaluator.evaluate_summaries(
            generated_summaries=valid_generated,
            reference_summaries=valid_reference,
            documents=valid_documents
        )
        
        evaluation_results[method_name] = eval_result
        
        # Print key metrics
        aggregate = eval_result['aggregate_scores']
        
        if 'rouge' in aggregate:
            rouge = aggregate['rouge']
            print(f"  ROUGE-1 F1: {rouge['rouge1_fmeasure_mean']:.4f}")
            print(f"  ROUGE-2 F1: {rouge['rouge2_fmeasure_mean']:.4f}")
            print(f"  ROUGE-L F1: {rouge['rougeL_fmeasure_mean']:.4f}")
        
        if 'overall' in aggregate:
            overall = aggregate['overall']
            print(f"  Overall Score: {overall['mean_score']:.4f} ± {overall['std_score']:.4f}")
    
    else:
        print(f"  No valid summaries for {method_name}")

## 5. Comprehensive Method Comparison

In [ ]:
# Compare all methods
if evaluation_results:
    comparison = evaluator.compare_methods(evaluation_results)
    
    print("\nComprehensive Method Comparison:")
    print("=" * 50)
    
    # Create comparison table
    comparison_data = []
    for method, rank in comparison['ranking'].items():
        scores = comparison['method_scores'][method]
        
        # Get ROUGE scores if available
        rouge_1 = 0
        if method in evaluation_results:
            agg_scores = evaluation_results[method]['aggregate_scores']
            if 'rouge' in agg_scores:
                rouge_1 = agg_scores['rouge']['rouge1_fmeasure_mean']
        
        comparison_data.append({
            'Method': method.title(),
            'Rank': rank,
            'Overall Score': f"{scores['mean']:.4f} ± {scores['std']:.4f}",
            'Mean Score': scores['mean'],
            'Std Score': scores['std'],
            'Min Score': scores['min'],
            'Max Score': scores['max'],
            'ROUGE-1 F1': rouge_1
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Rank')
    
    # Display table
    display(comparison_df[['Rank', 'Method', 'Overall Score', 'ROUGE-1 F1']])
    
    # Create visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Overall scores comparison
    methods = comparison_df['Method']
    means = comparison_df['Mean Score']
    stds = comparison_df['Std Score']
    
    bars1 = ax1.bar(methods, means, yerr=stds, capsize=5, alpha=0.7)
    ax1.set_title('Overall Scores Comparison')
    ax1.set_ylabel('Score')
    ax1.set_ylim(0, max(means) + 0.1)
    
    for bar, mean in zip(bars1, means):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{mean:.3f}', ha='center', va='bottom')
    
    # ROUGE-1 comparison
    rouge_scores = comparison_df['ROUGE-1 F1']
    bars2 = ax2.bar(methods, rouge_scores, alpha=0.7, color='orange')
    ax2.set_title('ROUGE-1 F1 Scores')
    ax2.set_ylabel('ROUGE-1 F1')
    ax2.set_ylim(0, max(rouge_scores) + 0.1)
    
    for bar, score in zip(bars2, rouge_scores):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom')
    
    # Score distribution (box plot)
    all_scores = []
    method_labels = []
    
    for method in evaluation_results.keys():
        individual_scores = [s['overall_score'] for s in evaluation_results[method]['individual_scores']]
        all_scores.append(individual_scores)
        method_labels.append(method.title())
    
    ax3.boxplot(all_scores, labels=method_labels)
    ax3.set_title('Score Distribution')
    ax3.set_ylabel('Score')
    ax3.tick_params(axis='x', rotation=45)
    
    # Performance radar chart
    categories = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Legal Acc', 'Readability']
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle
    
    ax4 = plt.subplot(2, 2, 4, projection='polar')
    
    for method in evaluation_results.keys():
        values = []
        agg_scores = evaluation_results[method]['aggregate_scores']
        
        # Get normalized scores for radar chart
        if 'rouge' in agg_scores:
            values.append(agg_scores['rouge']['rouge1_fmeasure_mean'])
            values.append(agg_scores['rouge']['rouge2_fmeasure_mean'])
            values.append(agg_scores['rouge']['rougeL_fmeasure_mean'])
        else:
            values.extend([0, 0, 0])
        
        if 'legal_accuracy' in agg_scores:
            values.append(agg_scores['legal_accuracy']['overall_accuracy_mean'])
        else:
            values.append(0)
        
        if 'readability' in agg_scores:
            values.append(agg_scores['readability']['readability_score_mean'] / 100)  # Normalize
        else:
            values.append(0)
        
        values += values[:1]  # Complete the circle
        
        ax4.plot(angles, values, 'o-', linewidth=2, label=method.title())
        ax4.fill(angles, values, alpha=0.25)
    
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories)
    ax4.set_ylim(0, 1)
    ax4.set_title('Performance Radar Chart')
    ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    
    plt.tight_layout()
    plt.show()

else:
    print("No evaluation results available. Please run the previous cells first.")

## 6. Detailed Error Analysis

In [ ]:
# Analyze worst and best performing examples
if evaluation_results:
    # Focus on the best performing method
    best_method = comparison['ranking'].items()[0][0]  # Get the top-ranked method
    best_results = evaluation_results[best_method]
    
    print(f"\nError Analysis for {best_method.title()} Method:")
    print("=" * 50)
    
    # Sort by overall score
    individual_scores = best_results['individual_scores']
    sorted_scores = sorted(enumerate(individual_scores), key=lambda x: x[1]['overall_score'])
    
    # Show worst performing examples
    print("\nWorst Performing Examples:")
    print("-" * 30)
    
    for i, (idx, score_data) in enumerate(sorted_scores[:3]):
        print(f"\nExample {i+1} (Score: {score_data['overall_score']:.3f}):")
        print(f"Generated length: {score_data['generated_length']} words")
        print(f"Reference length: {score_data['reference_length']} words")
        
        # Show quality scores if available
        if 'quality_assessment' in score_data:
            qa = score_data['quality_assessment']
            print(f"Coverage: {qa.get('coverage', 0):.3f}")
            print(f"Readability: {qa.get('readability', 0):.3f}")
            print(f"Legal Coherence: {qa.get('legal_coherence', 0):.3f}")
        
        # Show truncated summaries
        gen_summary = results[best_method][idx]
        ref_summary = test_summaries[idx]
        
        print(f"Generated: {gen_summary[:200]}...")
        print(f"Reference: {ref_summary[:200]}...")
    
    # Show best performing examples
    print("\n\nBest Performing Examples:")
    print("-" * 30)
    
    for i, (idx, score_data) in enumerate(sorted_scores[-3:]):
        print(f"\nExample {i+1} (Score: {score_data['overall_score']:.3f}):")
        print(f"Generated length: {score_data['generated_length']} words")
        print(f"Reference length: {score_data['reference_length']} words")
        
        # Show quality scores if available
        if 'quality_assessment' in score_data:
            qa = score_data['quality_assessment']
            print(f"Coverage: {qa.get('coverage', 0):.3f}")
            print(f"Readability: {qa.get('readability', 0):.3f}")
            print(f"Legal Coherence: {qa.get('legal_coherence', 0):.3f}")
        
        # Show truncated summaries
        gen_summary = results[best_method][idx]
        ref_summary = test_summaries[idx]
        
        print(f"Generated: {gen_summary[:200]}...")
        print(f"Reference: {ref_summary[:200]}...")

else:
    print("No evaluation results available for error analysis.")

## 7. Legal Entity Preservation Analysis

In [ ]:
# Analyze legal entity preservation across methods
if evaluation_results:
    print("Legal Entity Preservation Analysis:")
    print("=" * 50)
    
    entity_preservation = {}
    
    for method_name, eval_result in evaluation_results.items():
        print(f"\n{method_name.title()} Method:")
        print("-" * 30)
        
        # Calculate entity preservation rates
        method_results = []
        
        for i, (doc, ref_summary, gen_summary) in enumerate(zip(test_documents, test_summaries, results[method_name])):
            if gen_summary.strip():
                # Extract entities
                ref_entities = legal_processor.extract_legal_entities(ref_summary)
                gen_entities = legal_processor.extract_legal_entities(gen_summary)
                
                # Calculate preservation for each entity type
                preservation_rates = {}
                for entity_type in ref_entities:
                    if ref_entities[entity_type]:
                        ref_set = set(ref_entities[entity_type])
                        gen_set = set(gen_entities.get(entity_type, []))
                        
                        preservation_rate = len(ref_set.intersection(gen_set)) / len(ref_set)
                        preservation_rates[entity_type] = preservation_rate
                
                method_results.append(preservation_rates)
        
        # Calculate average preservation rates
        avg_preservation = defaultdict(list)
        for result in method_results:
            for entity_type, rate in result.items():
                avg_preservation[entity_type].append(rate)
        
        entity_preservation[method_name] = {
            entity_type: np.mean(rates) for entity_type, rates in avg_preservation.items()
        }
        
        # Print preservation rates
        for entity_type, avg_rate in entity_preservation[method_name].items():
            print(f"  {entity_type.title()}: {avg_rate:.3f}")
    
    # Create visualization
    if entity_preservation:
        # Get all entity types
        all_entity_types = set()
        for method_data in entity_preservation.values():
            all_entity_types.update(method_data.keys())
        
        # Create heatmap data
        heatmap_data = []
        methods_list = list(entity_preservation.keys())
        
        for method in methods_list:
            row = []
            for entity_type in all_entity_types:
                row.append(entity_preservation[method].get(entity_type, 0))
            heatmap_data.append(row)
        
        # Create heatmap
        plt.figure(figsize=(12, 8))
        
        heatmap_df = pd.DataFrame(
            heatmap_data,
            index=[m.title() for m in methods_list],
            columns=[et.title() for et in all_entity_types]
        )
        
        sns.heatmap(heatmap_df, annot=True, cmap='YlOrRd', fmt='.3f', 
                   cbar_kws={'label': 'Preservation Rate'})
        plt.title('Legal Entity Preservation Rates Across Methods')
        plt.xlabel('Entity Type')
        plt.ylabel('Summarization Method')
        plt.tight_layout()
        plt.show()

else:
    print("No evaluation results available for entity analysis.")

## 8. Citation Coverage Analysis

In [ ]:
# Analyze citation coverage
if evaluation_results:
    print("Citation Coverage Analysis:")
    print("=" * 30)
    
    citation_analysis = {}
    
    for method_name, eval_result in evaluation_results.items():
        print(f"\n{method_name.title()} Method:")
        print("-" * 20)
        
        # Calculate citation coverage rates
        coverage_rates = []
        total_citations = 0
        covered_citations = 0
        
        for i, (doc, gen_summary) in enumerate(zip(test_documents, results[method_name])):
            if gen_summary.strip():
                # Parse citations
                doc_citations = legal_processor.parse_legal_citations(doc)
                gen_citations = legal_processor.parse_legal_citations(gen_summary)
                
                doc_citation_set = set(c['full_citation'] for c in doc_citations)
                gen_citation_set = set(c['full_citation'] for c in gen_citations)
                
                if doc_citation_set:
                    coverage = len(doc_citation_set.intersection(gen_citation_set)) / len(doc_citation_set)
                    coverage_rates.append(coverage)
                    total_citations += len(doc_citation_set)
                    covered_citations += len(doc_citation_set.intersection(gen_citation_set))
        
        if coverage_rates:
            avg_coverage = np.mean(coverage_rates)
            overall_coverage = covered_citations / total_citations if total_citations > 0 else 0
            
            citation_analysis[method_name] = {
                'average_coverage': avg_coverage,
                'overall_coverage': overall_coverage,
                'total_document_citations': total_citations,
                'total_covered_citations': covered_citations
            }
            
            print(f"  Average Coverage: {avg_coverage:.3f}")
            print(f"  Overall Coverage: {overall_coverage:.3f}")
            print(f"  Total Document Citations: {total_citations}")
            print(f"  Total Covered Citations: {covered_citations}")
        else:
            print(f"  No citations found for analysis")
    
    # Create visualization
    if citation_analysis:
        methods = list(citation_analysis.keys())
        avg_coverages = [citation_analysis[m]['average_coverage'] for m in methods]
        overall_coverages = [citation_analysis[m]['overall_coverage'] for m in methods]
        
        x = np.arange(len(methods))
        width = 0.35
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        bars1 = ax.bar(x - width/2, avg_coverages, width, label='Average Coverage', alpha=0.7)
        bars2 = ax.bar(x + width/2, overall_coverages, width, label='Overall Coverage', alpha=0.7)
        
        ax.set_xlabel('Summarization Method')
        ax.set_ylabel('Citation Coverage Rate')
        ax.set_title('Citation Coverage Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels([m.title() for m in methods])
        ax.legend()
        ax.set_ylim(0, 1)
        
        # Add value labels on bars
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2, height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()

else:
    print("No evaluation results available for citation analysis.")

## 9. Generate Evaluation Report

In [ ]:
# Generate comprehensive evaluation report
if evaluation_results:
    report = evaluator.generate_evaluation_report({
        'aggregate_scores': {
            method: eval_result['aggregate_scores'] 
            for method, eval_result in evaluation_results.items()
        },
        'method_comparison': comparison if 'comparison' in locals() else {}
    })
    
    # Save report to file
    with open('../data/processed_data/evaluation_report.txt', 'w', encoding='utf-8') as f:
        f.write(report)
    
    print("Evaluation report generated!")
    print("Report saved to: ../data/processed_data/evaluation_report.txt")
    
    # Display first part of report
    print("\nEvaluation Report Preview:")
    print("=" * 40)
    print(report[:1000] + "..." if len(report) > 1000 else report)

else:
    print("No evaluation results available for report generation.")

## 10. Save Complete Evaluation Results

In [ ]:
# Save all evaluation results
complete_results = {
    'test_documents': test_documents,
    'test_summaries': test_summaries,
    'generated_summaries': results,
    'evaluation_results': evaluation_results,
    'method_comparison': comparison if 'comparison' in locals() else {},
    'entity_preservation': entity_preservation if 'entity_preservation' in locals() else {},
    'citation_analysis': citation_analysis if 'citation_analysis' in locals() else {},
    'test_size': test_size
}

# Save to JSON
with open('../data/processed_data/complete_evaluation_results.json', 'w', encoding='utf-8') as f:
    json.dump(complete_results, f, indent=2, ensure_ascii=False, default=str)

print("Complete evaluation results saved to: ../data/processed_data/complete_evaluation_results.json")

# Also save method comparison as CSV
if 'comparison' in locals() and comparison:
    comparison_df = pd.DataFrame([
        {
            'method': method,
            'rank': comparison['ranking'][method],
            'mean_score': scores['mean'],
            'std_score': scores['std'],
            'min_score': scores['min'],
            'max_score': scores['max']
        }
        for method, scores in comparison['method_scores'].items()
    ])
    
    comparison_df.to_csv('../data/processed_data/final_method_comparison.csv', index=False)
    print("Method comparison saved to: ../data/processed_data/final_method_comparison.csv")

print("\nEvaluation completed successfully!")
print("All results have been saved to the processed_data directory.")

## Summary

This comprehensive evaluation notebook provided:

### 1. **Multi-Method Evaluation**
- Evaluated extractive, abstractive, and hybrid summarization methods
- Used domain-specific metrics tailored for legal documents
- Provided detailed performance analysis and comparison

### 2. **Domain-Specific Metrics**
- **ROUGE Scores**: Standard n-gram overlap metrics
- **BERTScore**: Semantic similarity assessment
- **Legal Accuracy**: Domain-specific accuracy measures
- **Citation Coverage**: Percentage of legal citations preserved
- **Entity Preservation**: Legal entity retention analysis

### 3. **Comprehensive Analysis**
- Performance comparison across methods
- Error analysis of best and worst performing examples
- Legal entity preservation analysis
- Citation coverage evaluation
- Visual representations of results

### 4. **Key Findings**
The evaluation revealed important insights about different summarization approaches:

- **Extractive methods** excel at preserving factual accuracy and legal citations
- **Abstractive methods** provide better readability and fluency
- **Hybrid methods** balance precision and readability effectively
- **Legal domain specificity** is crucial for accurate evaluation

### 5. **Practical Implications**
- Method selection depends on specific use case requirements
- Trade-offs between accuracy and readability need careful consideration
- Domain-specific evaluation provides more meaningful insights
- Legal entity and citation preservation are critical quality indicators

This evaluation framework provides a robust foundation for assessing legal document summarization systems and can be extended to include additional metrics or adapted for other legal domains.